# 업태·종목 기반 자금종류 추천 시스템 (BGE-M3 & Qdrant)

이 노트북은 BGE-M3 임베딩과 Qdrant 벡터 데이터베이스를 사용하여 업태와 종목을 입력하면 적합한 자금종류를 추천하는 시스템을 구현합니다.

## Step 1: Pull and Run Qdrant Docker Image

In [ ]:
# No Docker needed for Colab! Qdrant will run in-memory mode.
print("✓ Ready to use Qdrant in-memory mode")

## Step 2: Install Required Python Packages

In [ ]:
%pip install -U FlagEmbedding
%pip install pandas
%pip install qdrant_client
%pip install tqdm
%pip install ipywidgets

## Step 3: Import Required Libraries

In [1]:
import pandas as pd
import json
from tqdm.notebook import tqdm
from FlagEmbedding import BGEM3FlagModel
from qdrant_client import QdrantClient, models

## Step 4: 자금 데이터셋 로드 (업태, 종목, 자금종류)

In [ ]:
# Load funding data from CSV file
def load_funding_data(file_path='funding_data.csv'):
    funding_df = pd.read_csv(file_path, sep='|')
    funding_json = funding_df.to_dict(orient='records')
    
    # Print the first record as JSON
    print(json.dumps(funding_json[0], indent=2, ensure_ascii=False))
    print(f"Total records: {len(funding_json)}")
    
    return funding_df, funding_json

funding_df, funding_json = load_funding_data()

## Step 5: Initialize the BGE-M3 model

In [3]:
def initialize_model():
    """Initialize the BGE-M3 embedding model"""
    return BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)

model = initialize_model()

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

## Step 6: 업태·종목 텍스트 포맷팅 및 임베딩 생성

In [ ]:
def create_business_text(record):
    """Format business information for embedding"""
    return f"업태: {record['업태']}\n종목: {record['종목']}"

# Let's take a single record as an example
sample_record = funding_df.iloc[0]
business_text = create_business_text(sample_record)

print("\n포맷된 업태·종목 텍스트:")
print(business_text)
print(f"\n해당하는 자금종류: {sample_record['자금종류']}")

In [ ]:
def generate_embeddings(text, model):
    """Generate all three types of embeddings for a text"""
    return model.encode(
        [text], 
        return_dense=True,
        return_sparse=True,
        return_colbert_vecs=True
    )

# Generate embeddings for the sample business record
output = generate_embeddings(business_text, model)

# Extract the embeddings
dense_vector = output['dense_vecs'][0]
sparse_weights = output['lexical_weights'][0]
colbert_vectors = output['colbert_vecs'][0]

### Displaying dense vectors

In [6]:
def display_dense_info(dense_vector):
    """Display information about dense vectors"""
    print("Dense vector information:")
    print(f"Shape: {dense_vector.shape}")
    print(f"First 5 elements: {dense_vector[:5]}")

display_dense_info(dense_vector)

Dense vector information:
Shape: (1024,)
First 5 elements: [-0.02007942  0.02020039 -0.04378937  0.02201727 -0.01971708]


### Displaying sparse vectors

In [7]:
def display_sparse_info(sparse_weights, model):
    """Display information about sparse vectors"""
    print("Sparse vector information:")
    print(f"Number of tokens with weights: {len(sparse_weights)}")
    
    # Convert token IDs to readable tokens
    token_map = model.convert_id_to_token([sparse_weights])
    
    # Display top tokens by weight
    print("\nTop 10 tokens by weight:")
    for token, weight in sorted(token_map.items(), key=lambda x: float(x[1]), reverse=True)[:10]:
        print(f"  {token}: {float(weight):.4f}")

display_sparse_info(sparse_weights, model)

Sparse vector information:
Number of tokens with weights: 57

Top 10 tokens by weight:
  vara: 0.3139
  shoes: 0.2366
  13: 0.2228
  weight: 0.2103
  speed: 0.2095
  y: 0.2015
  Product: 0.1993
  Sho: 0.1895
  Kin: 0.1848
  Sau: 0.1793


### Displaying ColBERT vectors

In [8]:
def display_colbert_info(colbert_vectors):
    """Display information about ColBERT vectors"""
    print(f"ColBERT vectors: {colbert_vectors.shape} (tokens × dimensions)")
    print("\nFirst 5 token vectors (first 3 dimensions each):")
    for i in range(min(5, len(colbert_vectors))):
        print(f"  Token {i}: {colbert_vectors[i][:3].tolist()}")

display_colbert_info(colbert_vectors)

ColBERT vectors: (82, 1024) (tokens × dimensions)

First 5 token vectors (first 3 dimensions each):
  Token 0: [-0.015244310721755028, -0.058750852942466736, 0.003742293221876025]
  Token 1: [-0.03496930003166199, -0.058924075216054916, -0.020842792466282845]
  Token 2: [-0.008677985519170761, -0.03177442401647568, -0.007166309282183647]
  Token 3: [-0.041619885712862015, -0.015667051076889038, -0.004003342241048813]
  Token 4: [-0.05215001106262207, -0.007239856291562319, 0.0014676559949293733]


## Step 7: 모든 업태·종목에 대한 임베딩 생성

In [ ]:
def generate_funding_embeddings(records, model):
    """Generate embeddings for all business records"""
    all_funding_embeddings = []
    
    # Process all records with progress bar
    for record in tqdm(records):
        # Format business text (업태 + 종목)
        business_text = create_business_text(record)
        
        # Generate embeddings
        output = generate_embeddings(business_text, model)
        
        # Store record and its embeddings
        funding_embedding = {
            "record": record,
            "dense_vector": output['dense_vecs'][0],
            "sparse_weights": output['lexical_weights'][0],
            "colbert_vectors": output['colbert_vecs'][0]
        }
        
        all_funding_embeddings.append(funding_embedding)
    
    print(f"Generated embeddings for {len(all_funding_embeddings)} records")
    return all_funding_embeddings

all_funding_embeddings = generate_funding_embeddings(funding_json, model)

## Step 8: Qdrant 컬렉션 생성 (Dense, Sparse, Multi-vectors)

In [ ]:
def create_qdrant_collection(collection_name="funding"):
    """Create Qdrant collection with appropriate vector configurations"""
    # Use in-memory mode for Colab (no Docker required)
    # Alternative: Use QdrantClient(path="./qdrant_data") for persistent storage
    client = QdrantClient(":memory:")
    
    # Create collection with dense vectors, sparse vectors, and ColBERT multi-vectors
    client.create_collection(
        collection_name=collection_name,
        vectors_config={
            "dense": models.VectorParams(
                size=1024,
                distance=models.Distance.COSINE
            ),
            "colbert": models.VectorParams(
                size=1024,
                distance=models.Distance.COSINE,
                multivector_config=models.MultiVectorConfig(
                    comparator=models.MultiVectorComparator.MAX_SIM
                ),
            )
        },
        sparse_vectors_config={
            "sparse": models.SparseVectorParams(
                index=models.SparseIndexParams(
                    on_disk=False  # Keep in memory for Colab
                )
            )
        },
    )
    
    print(f"Collection '{collection_name}' created successfully in-memory")
    return client

client = create_qdrant_collection()

## Step 9: Convert BGE-M3 sparse output to Qdrant format

In [15]:
def create_sparse_vector(sparse_data):
    """Convert BGE-M3 sparse output to Qdrant sparse vector format"""
    sparse_indices = []
    sparse_values = []
    
    for key, value in sparse_data.items():
        # Only process positive values
        if float(value) > 0:
            # Handle string keys
            if isinstance(key, str):
                if key.isdigit():
                    key = int(key)
                else:
                    continue
                
            sparse_indices.append(key)
            sparse_values.append(float(value))
    
    return models.SparseVector(
        indices=sparse_indices,
        values=sparse_values
    )

## Step 10: Qdrant 컬렉션에 자금 데이터 삽입

In [ ]:
def insert_funding_to_qdrant(client, funding_embeddings, collection_name="funding"):
    """Insert funding embeddings into Qdrant collection"""
    for embedding in tqdm(funding_embeddings):
        record = embedding["record"]
        dense_vector = embedding["dense_vector"]
        colbert_vectors = embedding["colbert_vectors"]
        sparse_data = embedding["sparse_weights"]

        # Convert sparse weights to Qdrant format
        qdrant_sparse = create_sparse_vector(sparse_data)
        
        # Insert into Qdrant
        client.upsert(
            collection_name=collection_name,
            points=[
                models.PointStruct(
                    id=record["Id"],
                    payload=record,
                    vector={
                        "dense": dense_vector,
                        "colbert": colbert_vectors,
                        "sparse": qdrant_sparse
                    }
                )
            ]
        )
    
    print(f"Successfully inserted {len(funding_embeddings)} records into the '{collection_name}' collection")

insert_funding_to_qdrant(client, all_funding_embeddings)

## Step 11: 업태·종목 입력 시 자금종류 추천 검색 함수

In [ ]:
def search_funding_types(client, model, business_type, business_item, limit=5, prefetch_limit=10, collection_name="funding"):
    """Search for funding types based on business type and item using hybrid search and reranking"""
    # Create search query from business type and item
    search_query = f"업태: {business_type}\n종목: {business_item}"
    
    # Generate embeddings for the query
    query_outputs = model.encode(
        [search_query],
        return_dense=True,
        return_sparse=True,
        return_colbert_vecs=True
    )
    
    dense_vec = query_outputs["dense_vecs"][0]
    sparse_vec = query_outputs["lexical_weights"][0]
    colbert_vec = query_outputs["colbert_vecs"][0]
    
    # Convert sparse vector to Qdrant format
    qdrant_sparse = create_sparse_vector(sparse_vec)
    
    # Set up prefetch for hybrid search
    prefetch = [
        models.Prefetch(
            query=qdrant_sparse,
            using="sparse",
            limit=prefetch_limit),
        models.Prefetch(
            query=dense_vec,
            using="dense",
            limit=prefetch_limit)
    ]
    
    # Perform reranking with ColBERT
    results = client.query_points(
        collection_name,
        prefetch=prefetch,
        query=colbert_vec,
        using="colbert",
        with_payload=True,
        limit=limit,
    )
    
    return results

## Step 12: 추천된 자금종류 결과 표시

In [ ]:
def display_funding_results(results, business_type, business_item):
    """Display funding type recommendations in a readable format"""
    print(f"입력: 업태='{business_type}', 종목='{business_item}'")
    print("=" * 60)
    print(f"\n추천 자금종류 ({len(results.points)}개):\n")
    
    # Collect all funding types
    funding_types_set = set()
    
    for i, result in enumerate(results.points):    
        record = result.payload
        print(f"{i+1}. 유사 업태/종목: {record['업태']} - {record['종목']}")
        print(f"   매칭 점수: {result.score:.2f}")
        print(f"   자금종류: {record['자금종류']}")
        
        # Add funding types to set
        funding_types = [ft.strip() for ft in record['자금종류'].split(',')]
        funding_types_set.update(funding_types)
        print()
    
    print("=" * 60)
    print(f"전체 추천 자금종류 키워드 ({len(funding_types_set)}개):")
    print(", ".join(sorted(funding_types_set)))

## Step 13: 예제 검색 테스트

In [ ]:
# 예제 1: 제조업 - 스마트폰 제조
result = search_funding_types(client, model, business_type="제조업", business_item="스마트폰 제조")
display_funding_results(result, "제조업", "스마트폰 제조")

In [ ]:
# 예제 2: 서비스업 - 한식 레스토랑
result = search_funding_types(client, model, business_type="서비스업", business_item="한식 레스토랑")
display_funding_results(result, "서비스업", "한식 레스토랑")

In [ ]:
# 예제 3: IT업 - 클라우드 플랫폼 개발
result = search_funding_types(client, model, business_type="IT업", business_item="클라우드 플랫폼 개발")
display_funding_results(result, "IT업", "클라우드 플랫폼 개발")

In [ ]:
# 예제 4: 농림어업 - 스마트팜 운영
result = search_funding_types(client, model, business_type="농림어업", business_item="스마트팜 운영")
display_funding_results(result, "농림어업", "스마트팜 운영")